# Railway Track Surface Defect Detection Training
## YOLOv8n on Railway Track Surface Defects Dataset (7 classes)

Run this notebook in Google Colab with **GPU runtime** (Runtime → Change runtime type → T4 GPU).
Training takes ~20 minutes for 100 epochs.

In [ ]:
# 1. Install dependencies
!pip install -q ultralytics roboflow

In [ ]:
# 2. Download dataset from Roboflow
from roboflow import Roboflow
import yaml

API_KEY = "oAHPVi0KEDHQXipmAaqd"
rf = Roboflow(api_key=API_KEY)
project = rf.workspace('project-hkopa').project('railway-track-surface-defects-m5e4e')
version = project.version(11)
dataset_path = version.download('yolov8', location='railway_track_dataset')
print(f'Dataset downloaded to: {dataset_path}')

In [ ]:
# 3. Split dataset into train/val/test (80/10/10)
import os
import shutil
import random
from pathlib import Path

random.seed(42)

DATA = Path('/content/railway_track_dataset')

# Collect all images + labels from train + valid
all_items = []
for split in ['train', 'valid']:
    img_dir = DATA / split / 'images'
    for img_path in sorted(img_dir.glob('*.jpg')):
        lbl_path = DATA / split / 'labels' / f'{img_path.stem}.txt'
        if lbl_path.exists():
            all_items.append((img_path, lbl_path))

print(f'Total labeled images: {len(all_items)}')
random.shuffle(all_items)

n = len(all_items)
splits = {
    'train': all_items[:int(n * 0.8)],
    'valid': all_items[int(n * 0.8):int(n * 0.9)],
    'test': all_items[int(n * 0.9):],
}

for name, items in splits.items():
    print(f'{name}: {len(items)} images')

# Create new directories and copy files
for name, items in splits.items():
    img_dir = DATA / name / 'images'
    lbl_dir = DATA / name / 'labels'
    if img_dir.parent.exists():
        shutil.rmtree(img_dir.parent)
    img_dir.mkdir(parents=True)
    lbl_dir.mkdir(parents=True)
    for img_path, lbl_path in items:
        shutil.copy2(img_path, img_dir / img_path.name)
        shutil.copy2(lbl_path, lbl_dir / lbl_path.name)

# Update data.yaml
yaml_path = DATA / 'data.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['train'] = '../train/images'
cfg['val'] = '../valid/images'
cfg['test'] = '../test/images'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Train/val/test split complete!')

In [ ]:
# 4. Train YOLOv8n (detection)
import torch
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/railway_track_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    project='railway_track_model',
    name='yolov8n',
    exist_ok=True,
    pretrained=True,
    optimizer='Adam',
    lr0=0.001,
    patience=20,
)
print('Training complete!')

In [ ]:
# 5. Evaluate on test set
metrics = model.val(data='/content/railway_track_dataset/data.yaml', split='test')
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

In [ ]:
# 6. Save best model for download
import os
import shutil

best_path = '/content/railway_track_model/yolov8n/weights/best.pt'
if os.path.exists(best_path):
    shutil.copy(best_path, '/content/railway_track_best.pt')
    print(f'Model saved to: /content/railway_track_best.pt')
    print(f'Size: {os.path.getsize(best_path) / 1e6:.1f} MB')
else:
    print(f'Not found at {best_path}')
    import glob
    for f in glob.glob('/content/railway_track_model/**/*.pt', recursive=True):
        print(f'Found: {f}')

## Download the model

1. Click the folder icon on the left sidebar in Colab
2. Navigate to `/content/railway_track_best.pt`
3. Right-click → Download
4. Place the downloaded file in your project as `models/rail_surface_yolov8n.pt`
5. Then run the webapp with `python -m webapp.main`

In [ ]:
# 7. Download directly from code
from google.colab import files
files.download('/content/railway_track_best.pt')